# Bird Diversity Data Cleaning and Merging
This notebook processes bird observation and climate data, merges them, cleans null values, and saves the final dataset.

## 1) Import libraries
We only need `pandas` for loading, merging, and cleaning the datasets.

In [49]:
import pandas as pd
import os

## 2) Load bird observation data
Load the bird observations dataset (`final3.csv`) and inspect its structure before merging.

In [35]:
# Load and inspect final3.csv
original_df = pd.read_csv('./final3.csv')
original_df.head()

,index,verbatimScientificName,stateProvince,individualCount,decimalLatitude,decimalLongitude,eventDate,avg_rad,cf_cvg,NDVI_raw,...,elevation_meters,BCSMASS,DUEXTT25,DUSMASS,OCSMASS,SO2SMASS,SO4SMASS,SSSMASS,TOTEXTTAU,TOTSCATAU
0,0,Anarhynchus alexandrinus,Mannar,8.0,9.058512,79.855020,2021-01-06,0.66,5.0,3792.0,...,0,5.794512e-10,0.001600,1.548033e-09,2.780749e-09,1.643020e-10,3.372161e-09,3.725563e-08,0.175115,0.165737
1,1,Columba livia,Colombo,31.0,6.927894,79.865005,2024-09-24,25.40,1.0,5692.0,...,12,3.096564e-10,0.021307,1.578878e-08,1.537093e-09,4.319662e-09,1.169898e-09,5.924096e-08,0.191869,0.184795
2,2,Hirundo rustica,Colombo,10.0,6.866285,79.931440,2024-12-23,10.16,7.0,7045.0,...,5,9.678071e-10,0.001877,2.117789e-09,4.676763e-09,4.725881e-09,4.424376e-09,3.249729e-08,0.198127,0.187366
3,3,Geokichla spiloptera,Matale,1.0,7.401229,80.690730,2024-09-13,1.13,1.0,9506.0,...,876,2.822976e-10,0.020004,1.525811e-08,1.535989e-09,2.551352e-09,1.286736e-09,3.795434e-08,0.162928,0.156176
4,4,Himantopus himantopus,Puttalam,6.0,8.154166,79.736084,2024-11-28,0.79,2.0,NaN,...,2,8.410959e-10,0.001956,1.985992e-09,4.197754e-09,1.223400e-09,4.910509e-09,4.148581e-08,0.213164,0.203239


In [36]:
original_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1999208 entries, 0 to 1999207
Data columns (total 21 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   index                   int64  
 1   verbatimScientificName  str    
 2   stateProvince           str    
 3   individualCount         float64
 4   decimalLatitude         float64
 5   decimalLongitude        float64
 6   eventDate               str    
 7   avg_rad                 float64
 8   cf_cvg                  float64
 9   NDVI_raw                float64
 10  LandCover_Class         int64  
 11  elevation_meters        int64  
 12  BCSMASS                 float64
 13  DUEXTT25                float64
 14  DUSMASS                 float64
 15  OCSMASS                 float64
 16  SO2SMASS                float64
 17  SO4SMASS                float64
 18  SSSMASS                 float64
 19  TOTEXTTAU               float64
 20  TOTSCATAU               float64
dtypes: float64(15), int64(3), str(3)
memory us

## 3) Load climate data
Load the climate dataset (`climate.csv`) that contains weather variables for the same locations and dates.

In [37]:
climate_df = pd.read_csv('./climate.csv')
climate_df.head()

,decimalLatitude,decimalLongitude,eventDate,provider,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_2m_mean,relative_humidity_2m_mean,shortwave_radiation_sum
0,9.058512,79.85502,2014-12-16,nasa_power,26.75,28.78,25.44,10.30,4.60,88.36,13.30
1,9.058512,79.85502,2018-02-20,nasa_power,25.80,30.39,22.37,0.00,4.89,68.25,23.98
2,9.058512,79.85502,2019-03-04,nasa_power,28.12,32.69,24.04,0.02,2.82,67.96,23.55
3,9.058512,79.85502,2019-03-17,nasa_power,29.01,33.55,25.35,0.03,4.20,68.33,23.27
4,9.058512,79.85502,2020-02-20,nasa_power,26.74,30.33,23.56,0.07,4.49,74.66,23.71


In [38]:
climate_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 80499 entries, 0 to 80498
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   decimalLatitude            80499 non-null  float64
 1   decimalLongitude           80499 non-null  float64
 2   eventDate                  80499 non-null  str    
 3   provider                   80499 non-null  str    
 4   temperature_2m_mean        80499 non-null  float64
 5   temperature_2m_max         80499 non-null  float64
 6   temperature_2m_min         80499 non-null  float64
 7   precipitation_sum          80499 non-null  float64
 8   wind_speed_2m_mean         80499 non-null  float64
 9   relative_humidity_2m_mean  80499 non-null  float64
 10  shortwave_radiation_sum    80499 non-null  float64
dtypes: float64(9), str(2)
memory usage: 6.8 MB


## 4) Clean and standardize climate columns
Remove unnecessary fields and rename climate columns to shorter, consistent names used in later steps.

In [39]:
climate_df = climate_df.drop(columns=["provider"])

In [40]:

climate_df = climate_df.rename(columns={
    "temperature_2m_mean": "temp_mean",
    "temperature_2m_max": "temp_max",
    "temperature_2m_min": "temp_min",
    "precipitation_sum": "rainfall",
    "wind_speed_2m_mean": "wind_mean",
    "relative_humidity_2m_mean": "humid_mean",
    "shortwave_radiation_sum": "shortwave_radiation"
})

## 5) Merge bird + climate datasets
Join climate variables to each bird observation using latitude, longitude, and date. A left join keeps all bird records and adds climate fields when available.

In [41]:
# Merge on latitude, longitude, and date
merged_df = pd.merge(original_df, climate_df, on=['decimalLatitude', 'decimalLongitude', 'eventDate'], how='left')

# Display merged dataframe
merged_df.head()

,index,verbatimScientificName,stateProvince,individualCount,decimalLatitude,decimalLongitude,eventDate,avg_rad,cf_cvg,NDVI_raw,...,SSSMASS,TOTEXTTAU,TOTSCATAU,temp_mean,temp_max,temp_min,rainfall,wind_mean,humid_mean,shortwave_radiation
0,0,Anarhynchus alexandrinus,Mannar,8.0,9.058512,79.855020,2021-01-06,0.66,5.0,3792.0,...,3.725563e-08,0.175115,0.165737,26.58,28.52,24.74,15.17,1.31,85.53,17.65
1,1,Columba livia,Colombo,31.0,6.927894,79.865005,2024-09-24,25.40,1.0,5692.0,...,5.924096e-08,0.191869,0.184795,27.52,29.55,26.03,2.63,5.50,84.41,20.77
2,2,Hirundo rustica,Colombo,10.0,6.866285,79.931440,2024-12-23,10.16,7.0,7045.0,...,3.249729e-08,0.198127,0.187366,26.21,28.84,24.10,0.25,2.51,84.30,20.49
3,3,Geokichla spiloptera,Matale,1.0,7.401229,80.690730,2024-09-13,1.13,1.0,9506.0,...,3.795434e-08,0.162928,0.156176,27.29,32.96,23.45,0.17,4.11,76.89,22.42
4,4,Himantopus himantopus,Puttalam,6.0,8.154166,79.736084,2024-11-28,0.79,2.0,NaN,...,4.148581e-08,0.213164,0.203239,25.86,27.55,24.85,0.99,6.15,84.07,8.78


In [42]:
merged_df.shape

(1999208, 28)

In [43]:
climate_df.shape

(80499, 10)

## 6) Handle missing climate values
After merging, some observations may not have matching climate data. Here we (1) check missing values and (2) drop rows where the key climate variables are missing.

In [44]:
merged_df.isna().sum()

index                         0
verbatimScientificName        0
stateProvince                 0
individualCount               0
decimalLatitude               0
decimalLongitude              0
eventDate                     0
avg_rad                       0
cf_cvg                        0
NDVI_raw                  81124
LandCover_Class               0
elevation_meters              0
BCSMASS                       0
DUEXTT25                      0
DUSMASS                       0
OCSMASS                       0
SO2SMASS                      0
SO4SMASS                      0
SSSMASS                       0
TOTEXTTAU                     0
TOTSCATAU                     0
temp_mean                   291
temp_max                    291
temp_min                    291
rainfall                    291
wind_mean                   291
humid_mean                  291
shortwave_radiation         291
dtype: int64

In [45]:
# Drop rows where any of the specified columns are null
cols_with_nulls = [
    'temp_mean', 'temp_max', 'temp_min', 'rainfall',
    'wind_mean', 'humid_mean', 'shortwave_radiation'
]
merged_df = merged_df.dropna(subset=cols_with_nulls)

In [46]:
merged_df.isnull().sum()

index                         0
verbatimScientificName        0
stateProvince                 0
individualCount               0
decimalLatitude               0
decimalLongitude              0
eventDate                     0
avg_rad                       0
cf_cvg                        0
NDVI_raw                  81102
LandCover_Class               0
elevation_meters              0
BCSMASS                       0
DUEXTT25                      0
DUSMASS                       0
OCSMASS                       0
SO2SMASS                      0
SO4SMASS                      0
SSSMASS                       0
TOTEXTTAU                     0
TOTSCATAU                     0
temp_mean                     0
temp_max                      0
temp_min                      0
rainfall                      0
wind_mean                     0
humid_mean                    0
shortwave_radiation           0
dtype: int64

In [47]:
merged_df.head()

,index,verbatimScientificName,stateProvince,individualCount,decimalLatitude,decimalLongitude,eventDate,avg_rad,cf_cvg,NDVI_raw,...,SSSMASS,TOTEXTTAU,TOTSCATAU,temp_mean,temp_max,temp_min,rainfall,wind_mean,humid_mean,shortwave_radiation
0,0,Anarhynchus alexandrinus,Mannar,8.0,9.058512,79.855020,2021-01-06,0.66,5.0,3792.0,...,3.725563e-08,0.175115,0.165737,26.58,28.52,24.74,15.17,1.31,85.53,17.65
1,1,Columba livia,Colombo,31.0,6.927894,79.865005,2024-09-24,25.40,1.0,5692.0,...,5.924096e-08,0.191869,0.184795,27.52,29.55,26.03,2.63,5.50,84.41,20.77
2,2,Hirundo rustica,Colombo,10.0,6.866285,79.931440,2024-12-23,10.16,7.0,7045.0,...,3.249729e-08,0.198127,0.187366,26.21,28.84,24.10,0.25,2.51,84.30,20.49
3,3,Geokichla spiloptera,Matale,1.0,7.401229,80.690730,2024-09-13,1.13,1.0,9506.0,...,3.795434e-08,0.162928,0.156176,27.29,32.96,23.45,0.17,4.11,76.89,22.42
4,4,Himantopus himantopus,Puttalam,6.0,8.154166,79.736084,2024-11-28,0.79,2.0,NaN,...,4.148581e-08,0.213164,0.203239,25.86,27.55,24.85,0.99,6.15,84.07,8.78


## 7) Export the final dataset
Save the cleaned, merged dataset to a CSV file for later analysis/modeling.

In [50]:
# Save to final4.csv
merged_df.to_csv('./final4.csv', index=False)
print(f'Saved updated final4.csv with {len(merged_df.columns)} columns:')
print(list(merged_df.columns))
print(f'\nFile size: {os.path.getsize("final4.csv") / 1e6:.1f} MB')

Saved updated final4.csv with 28 columns:
['index', 'verbatimScientificName', 'stateProvince', 'individualCount', 'decimalLatitude', 'decimalLongitude', 'eventDate', 'avg_rad', 'cf_cvg', 'NDVI_raw', 'LandCover_Class', 'elevation_meters', 'BCSMASS', 'DUEXTT25', 'DUSMASS', 'OCSMASS', 'SO2SMASS', 'SO4SMASS', 'SSSMASS', 'TOTEXTTAU', 'TOTSCATAU', 'temp_mean', 'temp_max', 'temp_min', 'rainfall', 'wind_mean', 'humid_mean', 'shortwave_radiation']

File size: 664.4 MB
